In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib as mpl
from matplotlib.ticker import AutoLocator, AutoMinorLocator, LogLocator
import glob
import cv2
import os
import re

In [ ]:
# Domain size in 3D index space
ncell = (192, 64, 64)
Lx = 8424000.0 # (R+0.2*R)*6 , R=1.17e6 cm
Ly = 2808000.0 # (R+0.2*R)*2 , R=1.17e6 cm
Lz = 2808000.0 # (R+0.2*R)*2 , R=1.17e6 cm

dxcell = Lx / ncell[0]
dycell = Ly / ncell[1]
dzcell = Lz / ncell[2]

R = 1170000.0 # cm

r_emitter_sphere_radius = R # cm
r_emitter_sphere_center_cm = ( 1425937.5, 1425937.5, 1425937.5 ) # cm
mu_emitter_sphere_MeV = -3.130432908491162 # MeV
opacity_emitter_sphere_inv_cm = 0.00000854544806011998 # inv cm
T_emitter_sphere_MeV = 5.1189 # MeV

r_shadowing_sphere_radius = 1.5*R/4.5 # cm
r_shadowing_sphere_center_cm = ( 4935937.5, 1425937.5, 1425937.5 )
mu_shadowing_sphere_MeV = -19.95627340541274 # MeV
opacity_shadowing_sphere_inv_cm = 0.00012391882339512724 # inv cm
T_shadowing_sphere_MeV = 0.0566631 # MeV

mu_vacuum_MeV = -25.859107462353197 # MeV
opacity_vacuum_inv_cm = 1.4877457720665111e-9 # inv cm
T_vacuum_MeV = 0.05666306 # MeV

In [ ]:
data_directory = '/pscratch/sd/b/bigqke/bigqke_scratch/emu_methods_paper_simulations/bigqke_per/shadow_test/shadow_test_ncell_192_64_64_dts_3.659e-07_ndir_1506_stored'

In [ ]:
# Angular discretization: unit momentum vectors phat = (pupx, pupy, pupz) / pupt.
# This run directory has no particle_input.dat. The same directions are on the
# plot particles (one energy). Integers are packed first, then AoS reals.
particle_header = os.path.join(data_directory, 'plt00768', 'neutrinos', 'Header')
with open(particle_header) as f:
    header_lines = [line.strip() for line in f]

n_real_extra = int(header_lines[2])
real_names = header_lines[3:3 + n_real_extra]
i = 3 + n_real_extra
i += 1  # n_int_extra
i += 1  # is_checkpoint
n_particles = int(header_lines[i]); i += 1
i += 2  # max_next_id, finest_level
n_grids = int(header_lines[i])
particles_per_grid = n_particles // n_grids

n_real = 3 + n_real_extra  # position plus extras
pup_slice = slice(3 + real_names.index('pupx'), 3 + real_names.index('pupz') + 1)
p0_index = 3 + real_names.index('pupt')

n_directions = n_particles // int(np.prod(ncell))
particle_file = os.path.join(data_directory, 'plt00768', 'neutrinos', 'Level_0', 'DATA_00000')
with open(particle_file, 'rb') as f:
    f.seek(particles_per_grid * 8)  # skip packed idcpu
    particle_reals = np.fromfile(f, dtype='<f8', count=n_directions * n_real).reshape(n_directions, n_real)

pup = particle_reals[:, pup_slice]
p0 = particle_reals[:, p0_index]
phat = pup / p0[:, None]

print(f'n_directions = {phat.shape[0]}')
print(f'p0 = {p0[0]}')
print(f'|phat| min/max = {np.linalg.norm(phat, axis=1).min()}, {np.linalg.norm(phat, axis=1).max()}')
